In [2]:
# Install required libraries
%pip install -q -U transformers accelerate bitsandbytes huggingface_hub

# Import libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
import os

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ============================================================
# Hugging Face Authentication
# ============================================================

import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except ImportError:
    # VS Code / Local Environment Fallback
    HF_TOKEN = os.getenv("HF_TOKEN", "")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found.")

login(token=HF_TOKEN)

NameError: name 'login' is not defined

In [3]:
# ============================================================
# Model Configuration
# ============================================================

# ONLY CHANGE THIS MODEL NAME
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"



In [4]:
# ============================================================
# Device Configuration
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print("Model:", MODEL_ID)


Device: cpu
Model: Qwen/Qwen2.5-1.5B-Instruct


In [5]:
# ============================================================
# Load Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)



In [ ]:
# ============================================================
# Load Model
# ============================================================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    token=HF_TOKEN
).to(device)

model.eval()

print("Model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


In [19]:
def chat(
    prompt,
    system_prompt="You are a helpful AI assistant.",
    max_new_tokens=200,
    temperature=0.7
):
    """
    Generate a response from a Hugging Face
    text-to-text/chat model.
    """

    # Create chat messages
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # Convert messages into the model's chat format
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize the formatted text
    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    # Move input tensors to the model device
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Remove the original prompt tokens
    input_length = inputs["input_ids"].shape[-1]

    generated_tokens = outputs[0][input_length:]

    # Decode generated response
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [17]:
response = chat(
    "What is Artificial Intelligence? Explain it in simple words."
)

print("AI:", response)

RuntimeError: Tensor.item() cannot be called on meta tensors

In [18]:
response = chat(
    "What is the difference between Machine Learning and Deep Learning?"
)

print("AI:", response)

RuntimeError: Tensor.item() cannot be called on meta tensors